# ViralTweets — Enriched Data Analysis

**Paper:** *Measuring and Detecting Virality on Social Media: The Case of Twitter's Viral Tweets Topic*  
Tuğrulcan Elmas, Selim Stephane, Célia Houssiaux — WWW '23 Companion, April 2023  
DOI: [10.1145/3543873.3587373](https://doi.org/10.1145/3543873.3587373) · arXiv: [2303.06120](https://arxiv.org/abs/2303.06120)

---

## Paper Context

Viral social media posts can spread misleading content at scale, making **early detection** critical. Prior work defined "virality" with self-chosen metrics (e.g. retweet count ≥ T) that may not reflect ground truth and can introduce many false positives. This paper is the first to use **Twitter's own "Viral Tweets" topic** as ground truth labels, then:

1. Benchmarks existing virality metrics against that ground truth;
2. Proposes a new metric (RT / Followers, threshold ≈ 2.16) with fewer false positives;
3. Trains a **BERTweet-based classifier** (F1 = 0.79) for early detection from tweet text + metadata only.

The code and tweet IDs are publicly available at [github.com/tugrulz/ViralTweets](https://github.com/tugrulz/ViralTweets).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PARQUET_FEATURES = 'classification/model_with_extra_features/final_dataset_since_october_2022.parquet.gzip'
PARQUET_LM       = 'classification/model_with_only_language_models/final_dataset_since_october_2022.parquet.gzip'
METRICS_CSV      = 'all_metric_stats.csv'

df      = pd.read_parquet(PARQUET_FEATURES)
df_lm   = pd.read_parquet(PARQUET_LM)
metrics = pd.read_csv(METRICS_CSV)
metrics = metrics[metrics['metric_name'] != 'unused']   # drop unused duplicate

print(f'Loaded {len(df):,} rows · {df.shape[1]} columns')

---
## 1. Dataset overview

In [ ]:
viral   = df['viral'].sum()
total   = len(df)
authors = df['author_id'].nunique()

print(f'Total tweets        : {total:>10,}')
print(f'  Viral (ground truth): {viral:>8,}  ({viral/total*100:.2f}%)')
print(f'  Non-viral           : {total-viral:>8,}  ({(total-viral)/total*100:.2f}%)')
print(f'Unique authors      : {authors:>10,}  (all went viral at least once)')
print(f'Languages           : {df["lang"].unique().tolist()}')
print(f'Date range          : {df["created_at"].min().date()} → {df["created_at"].max().date()}')
print()
print('Columns:', list(df.columns))

### Data collection design

- **Viral set** (`data/viral.csv`): 1,042 tweet IDs scraped from Twitter's "Viral Tweets" topic page (Oct–Nov 2022).  
  Only 1,008 were successfully hydrated via the API.
- **Control set** (`data/control.csv`): ~1.14 M tweet IDs — the full timelines of the same 814 authors who appeared in the viral set.  
  This design ensures the classifier learns *what makes a tweet viral for a given author*, not simply *which users go viral*.
- The two parquet files are nearly identical; the only difference is the `urls` column (different pre-processing in each pipeline).

---
## 2. Temporal distribution of viral tweets

In [ ]:
df['date'] = df['created_at'].dt.date
daily = df.groupby('date').agg(total=('id','count'), viral=('viral','sum'))
daily['viral_pct'] = daily['viral'] / daily['total'] * 100

print('Days with viral tweets:')
print(daily[daily['viral'] > 0][['total','viral','viral_pct']].to_string())

**Key observation:** Viral tweets cluster in **early-to-mid November 2022** — exactly when Elon Musk completed the Twitter acquisition (Oct 27) and the subsequent period of mass layoffs, policy changes, and public controversy (Nov 4–20). This suggests virality in the dataset is strongly driven by a **single real-world event cycle**, which may limit the model's generalisability to other periods.

---
## 3. Engagement metrics — viral vs non-viral

In [ ]:
engagement = ['retweet_count', 'reply_count', 'like_count', 'quote_count']
g = df.groupby('viral')[engagement].agg(['median','mean','max'])
g.index = ['Non-viral', 'Viral']
print(g.to_string())

**Note:** Engagement metrics are available *after* a tweet has already gone viral, so they cannot be used for *early detection*. They establish the upper-bound signal and motivate threshold-based detection metrics.

---
## 4. Content & author features

In [ ]:
print('--- Median values (can be observed at tweet time) ---')
num_feats = ['tweet_length','nb_of_hashtags','nb_of_mentions',
             'followers_count','following_count','tweet_count','sentiment_score']
print(df.groupby('viral')[num_feats].median()
        .rename(index={False:'Non-viral', True:'Viral'}).T.to_string())

print()
print('--- Boolean features (% True) ---')
for col in ['possibly_sensitive','has_media','verified']:
    v  = df[df['viral']][col].mean()*100
    nv = df[~df['viral']][col].mean()*100
    print(f'  {col:<22}: Viral {v:.1f}%   Non-viral {nv:.1f}%')

print()
print('--- Sentiment distribution ---')
sent = df.groupby(['viral','sentiment']).size().unstack(fill_value=0)
sent_pct = sent.div(sent.sum(axis=1), axis=0)*100
sent_pct.index = ['Non-viral','Viral']
print(sent_pct.round(1))

**Key findings (statistically significant per paper, p < 0.05 except 'Contains Mentions'):**

| Feature | Viral | Non-viral | Interpretation |
|---|---|---|---|
| `has_media` | **62%** | 19% | Strongest binary predictor — images/videos drive sharing |
| Sentiment (NEGATIVE) | **74%** | 61% | Emotional/negative content spreads more readily |
| `tweet_length` (median) | **72 chars** | 50 chars | Viral tweets are more substantive |
| `nb_of_mentions` (median) | **0** | 1 | Viral tweets address the public, not individuals |
| `followers_count` (median) | 16,923 | 14,704 | Modest difference — not a strong standalone predictor |
| `verified` | 5.5% | 5.0% | Near-identical — verification status barely matters |

The paper notes that `nb_of_mentions` differences are **not** statistically significant (p ≥ 0.05) despite the median difference above.

---
## 5. Virality metric benchmarks

The paper evaluates metrics against the 1,008 ground-truth viral tweets and a theoretical pool of ~1.36 M control tweets.

**Metrics tested:**

| Name | Formula | Requires |
|---|---|---|
| RT > T | retweet_count ≥ T (fixed threshold) | nothing |
| RT > Avg. RT | RT / mean(user's past RTs) ≥ threshold | user timeline |
| RT > Med. RT | RT / median(user's past RTs) ≥ threshold | user timeline |
| RT Percentile | RT ≥ k-th percentile of user's RTs | user timeline |
| RT / Followers | RT / followers_count ≥ threshold | follower count only |
| log(RT / Followers) | log(RT) / log(followers) ≥ threshold | follower count only |
| log(RT) / Followers | log(RT) / followers ≥ threshold | follower count only |
| RT / log(Followers) | RT / log(followers) ≥ threshold | follower count only |
| Influence Score | RoBERTa-based score ≥ threshold | none (text only) |

**Two AUC variants:**
- **AUC-1**: standard ROC-AUC over all FPR values
- **AUC-95** ("roc95" in code): ROC-AUC constrained to FPR ∈ [0, 0.016], rescaled to [0,1] — focuses on the operationally realistic low-FPR region
- **Harmonic**: harmonic mean of AUC-1 and AUC-95 — the paper's recommended ranking criterion

In [ ]:
# AUC summary table
summary = (metrics.groupby('metric_name')
           .agg(AUC_1=('roc1','first'), AUC_95=('roc95','first'),
                Harmonic=('harmonic','first'), Best_F1=('f1','max'))
           .sort_values('Harmonic', ascending=False))
print(summary.round(4).to_string())

In [ ]:
# False positives at TPR ~ 0.95  (operational cost of high recall)
print('False positives when catching ~95% of viral tweets:')
print(f'{"Metric":<25} {"TPR":>6} {"FP":>9} {"Threshold":>12}')
print('-' * 58)
for name, grp in metrics.groupby('metric_name'):
    near = grp[grp['tpr'] >= 0.95].nsmallest(1, 'tpr')
    if len(near):
        r = near.iloc[0]
        print(f'{name:<25} {r.tpr:>6.3f} {r.fp:>9,.0f} {r.threshold:>12.2f}')

In [ ]:
# Best F1 operating point per metric
print('Best F1 operating point per metric:')
print(f'{"Metric":<25} {"F1":>6} {"Prec":>6} {"Recall":>7} {"FP":>8} {"Threshold":>12}')
print('-' * 70)
for name, grp in metrics.groupby('metric_name'):
    best = grp.loc[grp['f1'].idxmax()]
    print(f'{name:<25} {best.f1:>6.4f} {best.precision:>6.3f} {best.recall:>7.3f} {best.fp:>8,.0f} {best.threshold:>12.2f}')

**Key findings:**

- **Influence Score** has the highest AUC-1 (0.961) but is **very lenient** — it needs 224,347 false positives to catch 95% of viral tweets, and its AUC-95 is only 0.700.
- **log(RT / Followers)** has the best harmonic AUC (0.874), balancing both AUC-1 (0.883) and AUC-95 (0.865). Only 11,067 FP @ 95% recall.
- **RT / Followers** (the paper's proposed threshold metric) ranks 2nd by harmonic AUC (0.866), produces only 13,040 FP @ 95% recall, and **does not require user timelines** — making it the most practical for real-time use.
- The paper finds a threshold of **RT / followers ≥ 2.16** maximises a balanced operating point; our data confirms precision ≈ 69% at this threshold.
- **RT Percentile** achieves the highest best-F1 (0.558) at its optimal operating point (threshold=1.0) with only 306 FP, though at the cost of lower AUC.
- Simple hard threshold **(RT > T)** is the weakest approach — requires very high absolute retweet counts, disfavouring smaller accounts.

---
## 6. The RT / Followers threshold (paper's proposed metric)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

df['rt_over_followers'] = df['retweet_count'] / (df['followers_count'] + 1)
y_true = df['viral'].astype(int)

print(f'RT/Followers at the paper\'s threshold (2.16):')
for t in [0.7, 1.19, 2.0, 2.16, 3.0]:
    pred = (df['rt_over_followers'] >= t).astype(int)
    f1 = f1_score(y_true, pred, zero_division=0)
    p  = precision_score(y_true, pred, zero_division=0)
    r  = recall_score(y_true, pred, zero_division=0)
    fp = int(pred.sum() - (pred & y_true).sum())
    print(f'  t={t:5.2f}  F1={f1:.4f}  P={p:.3f}  R={r:.3f}  FP={fp:,}')

print()
print('Note: t=0.70 maximises F1 on this dataset; t=2.16 (paper) prioritises precision (fewer FP).')

The paper chose **t ≈ 2.16** not to maximise F1 but to **minimise false positives** while maintaining reasonable recall — an important distinction for real-world deployment where false alarms are costly.

---
## 7. Classification results

**Training design:**
- Only tweets from the same author on the same day as a viral tweet are used (reduces confounder of author identity).
- 80/20 train/test split, stratified; test set **balanced** (equal viral/non-viral).
- Two settings: (A) language model only; (B) language model + 7 extra features.
- **Extra features:** `verified`, `tweet_length`, `possibly_sensitive`, `sentiment`, `nb_of_hashtags`, `has_media`, `nb_of_mentions`
- Optimizer: AdamW, lr=5e-5; 15 epochs; batch size=32; loss: BCEWithLogitsLoss.

In [ ]:
results = pd.DataFrame({
    'Model': ['BERTweet (vinai/bertweet-base)', 'BERT-tiny (prajjwal1/bert-tiny)',
              'BERT-base-cased', 'RoBERTa-base'],
    'LM-only Acc.':  [0.748, 0.729, 0.694, 0.742],
    'LM-only F1':    [0.766, 0.755, 0.714, 0.764],
    'LM-only Prec.': [0.717, 0.689, 0.670, 0.704],
    'LM-only Rec.':  [0.822, 0.834, 0.764, 0.834],
    '+Features Acc.':  [0.777, 0.723, 0.704, 0.729],
    '+Features F1':    [0.793, 0.762, 0.734, 0.761],
    '+Features Prec.': [0.740, 0.668, 0.667, 0.682],
    '+Features Rec.':  [0.854, 0.885, 0.815, 0.860],
}).set_index('Model')

print(results.to_string())
print()
print('Delta (+Features vs LM-only):')
delta = results[['+Features Acc.','+Features F1']] - results[['LM-only Acc.','LM-only F1']].values
delta.columns = ['ΔAcc.','ΔF1']
print(delta.round(3).to_string())

**Key findings:**

- **BERTweet** is the best model in both settings because it was pre-trained specifically on tweets (vocabulary, emoji handling, normalisation).
- Adding the 7 extra features yields a **consistent +2–3 pp improvement** in accuracy and F1 across all models.
- `has_media` is likely the dominant extra feature given its 3× rate difference between viral/non-viral.
- The balanced test set makes these accuracy figures interpretable as balanced accuracy — a more meaningful metric than raw accuracy given the 0.2% base rate.

---
## 8. What can be done — research and engineering extensions

### 8.1 Improve the classifier

| Idea | Rationale | Effort |
|---|---|---|
| **Multi-modal input** | 62% of viral tweets have media; image/video embeddings (CLIP, VideoMAE) would add strong signal | High |
| **Larger tweet LMs** | Twitter-RoBERTa-large, TweetEval suite, or instruction-tuned LLMs (fine-tuned LLaMA 3) | Medium |
| **Temporal / contextual features** | Hour-of-day (pattern visible in data), day-of-week, time since account creation, recent engagement velocity | Low |
| **Graph features** | Author's PageRank / centrality in follower graph; viral tweets originate disproportionately from hub nodes | High |
| **Calibrated probability output** | Current models produce scores, not calibrated P(viral); Platt scaling / isotonic regression needed for production use | Low |
| **Focal loss / asymmetric sampling** | 0.2% positive rate; focal loss or hard-example mining could further help vs. class balancing | Low |
| **Hybrid metric + LM ensemble** | Combine best metric (log(RT/Followers), AUC-harmonic 0.874) with BERTweet score for a two-stage filter | Medium |

### 8.2 Strengthen the metric analysis

| Idea | Rationale |
|---|---|
| **Composite metric** | Combine top two metrics (log(RT/Followers) + RT/Followers) — both are low-FP and don't need timelines |
| **Per-author threshold adaptation** | `retweet_count_user_viral_threshold` already computed per user; personalised thresholds could beat global ones |
| **Real-time metric** | All high-harmonic metrics (log(RT/Followers), RT/Followers) are computable at tweet time — implement a streaming detector |

### 8.3 Extend the dataset

| Idea | Rationale |
|---|---|
| **Longer time window** | Dataset covers only 7 weeks, heavily biased toward the Elon Musk/Twitter acquisition news cycle; broader period would test generalisability |
| **Multi-language** | All 432,687 tweets are English; Twitter's Viral Tweets topic includes non-English content |
| **Cross-platform** | The RT/Followers metric is designed to generalise to any platform with equivalent signals (Instagram likes/followers, etc.) |
| **Post-API-change data** | Twitter's API access changed drastically in 2023; re-collecting data post-change would test robustness |

### 8.4 Address limitations the paper identifies

| Limitation | Mitigation |
|---|---|
| **RT/Followers favours small accounts** | A high follower account needs proportionally more RTs, which may be unfair; a log-normalized variant (log(RT/Followers)) partially fixes this |
| **Topic-specific bias** | Scraping only the "Viral Tweets" topic means the 1,008 positives may not represent all forms of virality (e.g. niche communities) |
| **Early detection framing** | Current models use features available at tweet time, but engagement velocity features (first 30 min RT rate) are not used |
| **Generative risk** | Paper notes: a good virality predictor could be used to *generate* viral content automatically — future work should include safety guardrails |

### 8.5 Engineering / reproducibility

| Idea | Details |
|---|---|
| **Experiment tracking** | Add MLflow or W&B logging to `classification.py` — currently results are written to `.txt` files |
| **One-command pipeline** | Makefile / DVC pipeline: hydrate → preprocess → train → evaluate |
| **Model export** | Serialize best BERTweet + features checkpoint to ONNX or HuggingFace Hub for inference |
| **Streaming demo** | A FastAPI service that accepts a tweet + author metadata and returns P(viral) using the trained model + RT/Followers rule |
| **Reproducible metric analysis** | `1-standardize_metrics.py` uses `glob('output_original/*.csv')` — the raw per-threshold CSV files are not committed; adding them would make the analysis fully reproducible |